## Demonstration of 3D LG-Mode collamation, imaging and M-Squared measurements

This notebook demonstrates the generation and analysis of a three-dimensional heat source, and solution to the heat equation using the Alternate Direction Implicit Crank-Nicolson discretization method of Douglas and Rachford.


## 1. Import libraries and generate heat sources
- Refer to NoteBook 1 for a refresher on importing modules.
- In this notebook we will see how the 3D methods inherit from their 2D ancestors.


<div class="alert alert-block alert-info"> **Info:** Here we create an instance of the 3D `photonicslab.heateqn.HeatEqn3D` object and use the `get_explicit_fd_params` method [overridden](https://docs.python.org/2/tutorial/classes.html "Python Classes Turorial") from the parent class `HeatEqn2D`.</div>

In [25]:
import os, sys
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.append(module_path)
    
import importlib
import photonicslab

importlib.reload(photonicslab.heateqn)
#mygauss3dQ = photonicslab.heateqn.HeatEqn3D(nx=110, ny=100, nz=150, nt=50, Lx=5e-3, Ly=5e-3, Lz=1e-1, \
#                                                    hdf5grp='nb8', cavity_mode_waist=191.5e-6)
mygauss3dQ = photonicslab.heateqn.HeatEqn3D(nx=1001, ny=1001, nz=50, nt=50, Lx=0.5, Ly=0.5, Lz=1., \
                                            Qmax=800, hdf5grp='nb8', cavity_mode_waist=191.5e-4)

importlib.reload(photonicslab.beamqual)
mym2test = photonicslab.beamqual.MSquared(wavelen=1.064e-4, waist=191.5e-4, flens=5e-1, p=0, l=0, z=-25, phi=0, \
                                         hdf5grp='nb8')
mym2test.plot_second_focal_plane(flens=25, zpts=100, z1=25, z2=25, savepath='images/nb8_250_2f.png')

ny:  1001
(1001, 1001, 100)


<img width="800" height="600" src="images/nb8_250_2f.png">

<div class="alert alert-block alert-success"> **Tip:** Many of the modules are built with exception handling. Should you try to visualize the cross-sectional heat profile of the crystal / lens beyond the extent of it's physical axial length, a `ValueError` exception will be raised. And remember to press `F5` to refresh the page if the image is not updating.</div>

## 2. Solve and Visualize 3D heat equation using various finite difference methods

- Create `FDSolver3D` object using the integration parameters configured with the `mygauss3dQ` object.
- Specify unique paths for thermal and source profiles in database.
- For comparison and benchmarking, configurations for the explicit and implicit methods are also prepared.

<div class="alert alert-block alert-danger"> **Warning:** If you've generated other profiles in corresponding notebooks for later analysis, ensure that you use different paths fo the thermal and phase profiles in the in the hierarchical data format `storage.hdf5` file.</div>

### 2.1. Alternating Direction Implicit Method

This is the optimal solution, and in the subsequent notebooks GPU-accelerated solutions of this implimentation will be explored.

In [14]:
importlib.reload(photonicslab.fdsolver)
myfdsolver3d = photonicslab.fdsolver.FDSolver3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')
myfdsolver3d.adi_method()

importlib.reload(photonicslab.visualize)
myview3d = photonicslab.visualize.Visualize3D(hdf5filename='hdf5db/storage.hdf5', hdf5grp='nb8')
myview3d.steadystate_profiles(savepath='images/nb8_steady_state.png')
myview3d.transient_profiles(savepath='videos/nb8_transient.mp4')

adi_method took 283.3801176548004 seconds.
transient_profiles took 47.18888807296753 seconds.


<img width="800" height="600" src="images/nb8_steady_state.png">
<video width="640" height="480" controls autoplay loop src="videos/nb8_transient.mp4"/>

### 2.2. Explicit Method

This method implements a direct brute force *nested for loop* approach. This is comparatively slow, and CPU intensive.